In [40]:
import cv2
import os
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
below_32px = []
min_px = 32

In [170]:
"""
    Downscale an image to a target resolution (below max_res), keeping aspect ratio.
    Returns the low-res and low-bitrate image image.
"""
def degrade_image(img, img_name, jpeg_quality=20, scale_factor=2, blur_kernel=3):
    h, w = img.shape[:2]

    # Simulate distant small object by downscaling
    new_w = max(1, int(w // scale_factor))
    new_h = max(1, int(h // scale_factor))

    # Resize down to simulate distant plate
    downscaled = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Resize up to original size (zoom effect)
    upscaled = cv2.resize(downscaled, (w, h), interpolation=cv2.INTER_LINEAR)

    # Resize down to simulate distant plate
    upscaled = cv2.resize(upscaled, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Add optional blur
    if blur_kernel and blur_kernel > 1:
        upscaled = cv2.GaussianBlur(upscaled, (blur_kernel, blur_kernel), 0)

    # Add JPEG compression artifacts
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality]
    result, encimg = cv2.imencode('.jpg', upscaled, encode_param)
    if result:
        degraded = cv2.imdecode(encimg, 1)
    else:
        degraded = upscaled  # fallback in case encoding fails

    return degraded

In [172]:
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
in_path = os.path.join(ROOT_DIR, 'data', 'all', 'experiment2', 'train', 'normal filtered 2')
out_path = os.path.join(ROOT_DIR, 'data', 'all', 'experiment2', 'train', 'low_qual filtered half comp')

files = os.listdir(in_path)

In [174]:
min_qual = 10
max_qual = 30

In [176]:
for image_name in files:
    img = cv2.imread(os.path.join(in_path, image_name))
    #res = random.randint(min_rs, max_rs)
    qual = random.randint(min_qual, max_qual)

    low_res_img = degrade_image(img, image_name, jpeg_quality=qual)

    if low_res_img is None:
        continue

    filename = f'{out_path}/{image_name}'
    cv2.imwrite(filename, low_res_img)
    #print(f'Saved low quality plate: {filename}')
    
    #plt.imshow(cv2.cvtColor(low_res_img, cv2.COLOR_BGR2RGB))
    #plt.axis('off')
    #plt.show()

In [22]:
from PIL import Image

def pixelate_image(image_path, output_path, pixelation_factor=2, quality=10):
    img = Image.open(image_path)
    # Shrink the image
    small_img = img.resize((img.width // pixelation_factor, img.height // pixelation_factor), Image.NEAREST)
    # Scale it back up using nearest neighbor resampling
    pixelated_img = small_img.resize(img.size, Image.NEAREST)
    pixelated_img.save(out_path, quality=quality, optimize=True)

In [38]:
for image_name in files:
    qual = random.randint(min_qual, max_qual)
    print(os.path.join(in_path, image_name))
    low_res_img = pixelate_image(os.path.join(in_path, image_name), image_name, 2, quality=qual)

    if low_res_img is None:
        continue

    filename = f'{out_path}/{image_name}'
    cv2.imwrite(filename, low_res_img)

C:\Thesis\LiPAD\data\all\experiment2\train\normal\1.jpg


ValueError: unknown file extension: 